# 3주차. RAG, Agentic RAG, ChromaDB

## 0. 목표

이번 주에는 수강생이 입력한 메모를 ChromaDB collection에 저장하고, agent가 필요한 순간 검색 tool을 호출해 답하게 만든다.

성취 기준:

- ChromaDB의 `client`, `collection`, `add`, `count`, `query` 흐름을 말할 수 있다.
- `hits`의 `content`, `metadata`, `distance`를 보고 검색 근거를 확인할 수 있다.
- 직접 검색 결과와 agent tool trace 안의 검색 결과를 비교할 수 있다.


## 1. 준비

로컬 `Jupyter Notebook` 또는 `JupyterLab`에서 repo 루트를 기준으로 실행한다.

모든 실습은 실제 OpenAI API를 호출한다. API 키와 모델 설정은 repo 루트의 `.env`에서 읽는다.

처음 읽는 순서: `docs/orientation.md` → 이 노트북 → `week03.py`의 `# TODO 문제`를 먼저 풀고 바로 아래 `# 모범 답안`과 비교한 뒤 Gradio 데모를 실행한다.


In [ ]:
# 흐름: repo 설정, 모델 helper, trace helper를 준비한다.
# 3주차에서는 답변보다 검색 hit와 tool trace가 더 중요한 관찰값이다.
import json
import os
import sys
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


REPO_ROOT = find_repo_root(Path.cwd())

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv(REPO_ROOT / ".env", override=True)

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("repo 루트 .env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace


## 2. 개념

오늘의 큰 그림: RAG는 먼저 검색한 내용을 모델 답변에 넣는 흐름이고, Agentic RAG는 모델이 검색이 필요한지 판단해 검색 tool을 호출하는 흐름이다. 이번 주의 검색 기억 저장소는 ChromaDB다.

반드시 이해할 것:

- embedding은 메모를 검색 가능한 형태로 바꾸기 위한 준비 단계다.
- ChromaDB `collection`은 문서, metadata, embedding 검색 결과를 함께 관리한다.
- `hits`는 답변의 근거 후보이며, 답변 문구보다 먼저 확인한다.
- agent가 검색 tool을 호출했는지는 trace로 검증한다.

지금은 몰라도 되는 것:

- vector database index 알고리즘
- embedding 거리 계산 수학
- ChromaDB HNSW 튜닝과 내부 저장 방식

막혔을 때 볼 trace: ChromaDB `count()`, `query(...)` 결과, `search_memory` tool call, tool result 안의 `hits`.


## 3. 기본 개념 실습

가장 작은 흐름은 수강생이 직접 입력한 메모를 ChromaDB persistent collection에 저장하는 것이다. 아직 agent를 붙이지 않고 `client`, `collection`, `add`, `count` 상태만 확인한다.


In [ ]:
# 흐름: ChromaDB collection을 만들고 학생 메모를 embedding으로 저장한다.
# count와 collection_name을 확인하면 검색할 기억이 준비됐는지 알 수 있다.
import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

student_memories = [
    "프로젝트 발표는 2026-04-24 10:00에 민수와 지아가 함께 진행한다.",
    "카나메이트 UI에서는 채팅 답변과 tool trace를 함께 보여준다.",
]

CHROMA_DIR = REPO_ROOT / "tmp" / "week03_chroma"
COLLECTION_NAME = "kanamate_week3_memories"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

embedding_function = OpenAIEmbeddingFunction(api_key=os.environ["OPENAI_API_KEY"], model_name=OPENAI_EMBEDDING_MODEL)
client = chromadb.PersistentClient(path=str(CHROMA_DIR), settings=Settings(anonymized_telemetry=False))
try:
    client.delete_collection(COLLECTION_NAME)
except Exception as exc:
    if "does not exist" not in str(exc).lower():
        raise

memory_collection = client.create_collection(name=COLLECTION_NAME, embedding_function=embedding_function)
memory_collection.add(
    ids=[f"memory-{index + 1}" for index in range(len(student_memories))],
    documents=student_memories,
    metadatas=[{"source": "student_input", "order": index + 1} for index, _ in enumerate(student_memories)],
)

collection_state = {
    "client_type": type(client).__name__,
    "persist_dir": str(CHROMA_DIR),
    "collection_name": memory_collection.name,
    "count": memory_collection.count(),
}
show_json(collection_state)


## 4. 카나메이트 확장 예제

저장된 메모 검색 결과를 수업 코드가 쓰기 쉬운 `search_memory_hits` 리스트로 정리하고, 같은 helper를 agent tool에서 재사용한다. Hit에는 `id`, `content`, `distance`, `metadata`를 모두 담는다.


In [ ]:
# 흐름: ChromaDB의 중첩 query 결과를 학생이 읽기 쉬운 hit 리스트로 바꾼다.
# 이 helper를 직접 호출한 결과와 agent tool 결과가 같은 흐름을 사용한다.
def format_chroma_results(found: dict[str, Any]) -> list[dict[str, Any]]:
    ids = found.get("ids", [[]])[0]
    documents = found.get("documents", [[]])[0]
    distances = found.get("distances", [[]])[0]
    metadatas = found.get("metadatas", [[]])[0]
    return [
        {
            "id": ids[index],
            "content": documents[index],
            "distance": distances[index],
            "metadata": metadatas[index] if index < len(metadatas) and metadatas[index] else {},
        }
        for index in range(len(ids))
    ]


def search_memory_hits(query: str, top_k: int = 2) -> list[dict[str, Any]]:
    """Return Chroma search results as a simple list of dictionaries."""
    found = memory_collection.query(query_texts=[query], n_results=top_k)
    return format_chroma_results(found)


@tool("search_memory", description="수강생이 입력한 메모를 ChromaDB와 OpenAI embedding 기반으로 검색한다.")
def search_memory(query: str, top_k: int = 2) -> str:
    """Search student memory with ChromaDB."""
    return json.dumps({"hits": search_memory_hits(query, top_k)}, ensure_ascii=False)


rag_agent = create_agent(
    model=make_model(700),
    tools=[search_memory],
    system_prompt="저장된 메모가 필요한 질문이면 search_memory 도구를 호출한 뒤, 찾은 근거를 바탕으로 답한다.",
)

student_question = "프로젝트 발표는 언제야?"
result = rag_agent.invoke({"messages": [{"role": "user", "content": student_question}]})

print(final_text(result))
show_json(extract_tool_trace(result))


## 5. 확장 예제 실행

같은 helper를 직접 호출해 hit 리스트를 보고, agent tool trace에도 같은 검색 결과가 들어가는지 확인한다. `collection_state`와 `hits`를 먼저 보고 최종 답변을 나중에 본다.


In [ ]:
# 흐름: 검색 helper를 직접 호출한 뒤, agent가 search_memory tool을 호출하는지도 확인한다.
# direct hits, agent answer, tool trace를 함께 봐야 RAG 흐름을 설명할 수 있다.
practice_question = "카나메이트 UI에서는 무엇을 함께 보여줘?"
practice_hits = search_memory_hits(practice_question, top_k=2)
practice_result = rag_agent.invoke({"messages": [{"role": "user", "content": practice_question}]})
practice_trace = extract_tool_trace(practice_result)

show_json(collection_state)
show_json(practice_hits)
print(final_text(practice_result))
show_json(practice_trace)

assert memory_collection.count() == len(student_memories)
assert collection_state["count"] == len(student_memories)
assert practice_hits
assert "metadata" in practice_hits[0]
assert any(event["event"] == "tool_call" and event["tool_name"] == "search_memory" for event in practice_trace)
assert any(event["event"] == "tool_result" and "hits" in event["content"] for event in practice_trace)
print("3주차 확장 예제 실행 통과")


## 6. 회고

개념 확인 질문:

1. RAG와 Agentic RAG는 검색을 누가, 언제 결정한다는 점에서 다른가?
2. ChromaDB `count()`와 `query(...)` 결과는 각각 무엇을 확인하는 값인가?
3. `hits`의 `content`와 최종 답변이 어긋나면 무엇을 의심해야 하는가?
4. `distance`는 왜 최종 답변 문구보다 먼저 봐야 하는가?

작은 응용 과제: 메모 한 줄을 의도적으로 바꾼 뒤 같은 질문을 실행하고, ChromaDB hit와 최종 답변이 어떻게 변하는지 비교한다.
